In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split,cross_validate,StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score,roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import ADASYN
from imblearn.pipeline import Pipeline

In [5]:
#read CSV file
df=pd.read_csv("bank-additional-full.csv",sep=';')

In [6]:
df=df.drop('duration',axis=1)

In [7]:
allColumns=df.select_dtypes(include='object').columns

In [8]:
print(df['job'].unique())

['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']


In [9]:
for col in allColumns:
  df=df.replace('unknown',df[col].mode()[0])

In [10]:
print(df['job'].unique())

['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'entrepreneur' 'student']


In [11]:
print(df['y'].unique())

['no' 'yes']


In [12]:
#label Encoding
df['y']=df['y'].map({'yes':1,'no':0})
print(df['y'].unique())

[0 1]


In [13]:
print(df.columns)

Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'campaign', 'pdays', 'previous',
       'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
       'euribor3m', 'nr.employed', 'y'],
      dtype='object')


In [14]:
#One hot encoding
df=pd.get_dummies(df,drop_first=True)
print(df.columns)

Index(['age', 'campaign', 'pdays', 'previous', 'emp.var.rate',
       'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y',
       'job_blue-collar', 'job_entrepreneur', 'job_housemaid',
       'job_management', 'job_retired', 'job_self-employed', 'job_services',
       'job_student', 'job_technician', 'job_unemployed', 'marital_divorced',
       'marital_married', 'marital_single', 'education_basic.4y',
       'education_basic.6y', 'education_basic.9y', 'education_high.school',
       'education_illiterate', 'education_professional.course',
       'education_university.degree', 'default_no', 'default_yes',
       'housing_no', 'housing_yes', 'loan_no', 'loan_yes', 'contact_telephone',
       'month_aug', 'month_dec', 'month_jul', 'month_jun', 'month_mar',
       'month_may', 'month_nov', 'month_oct', 'month_sep', 'day_of_week_mon',
       'day_of_week_thu', 'day_of_week_tue', 'day_of_week_wed',
       'poutcome_nonexistent', 'poutcome_success'],
      dtype='object')


In [19]:
x=df.drop('y',axis=1)
y=df['y']

In [20]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
print(x_train['age'].count())

32950


In [21]:
scalar=StandardScaler()
scalar.fit_transform(x_train)
scalar.transform(x_test)

array([[-0.77033007,  0.88631588,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [-0.28972159, -0.56702251,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [ 3.17065947, -0.20368791,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       ...,
       [-0.67420837, -0.56702251,  0.19658384, ..., -0.4964409 ,
        -2.50346033, -0.18627755],
       [ 0.38313029,  1.61298507,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [ 0.19088689,  0.88631588,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755]])

In [39]:
adasyn=ADASYN(random_state=4)
def evaluates_model(x,y,sampler,label):
  model=DecisionTreeClassifier(random_state=42)
  pipeline=Pipeline([('sampler',sampler),('model',model)])
  scoring=(['accuracy','precision','recall','f1','roc_auc'])
  skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
  results=cross_validate(pipeline,x,y,cv=skf,scoring=scoring)

  print(f"==={label}====")
  print("Accuracy :",results['test_accuracy'].mean())
  print("precision :",results['test_precision'].mean())
  print("recall :",results['test_recall'].mean())
  print("f1 :",results['test_f1'].mean())
  print("roc_auc :",results['test_roc_auc'].mean())

In [40]:
evaluates_model(x_train,y_train,sampler=ADASYN(random_state=42),label="Adasyn")
evaluates_model(x_train,y_train,sampler='passthrough',label='Baseline')

===Adasyn====
Accuracy : 0.8321092564491656
precision : 0.29244373495778764
recall : 0.3445614595161308
f1 : 0.31629362358628843
roc_auc : 0.6216432674531533
===Baseline====
Accuracy : 0.8383004552352048
precision : 0.30334360646133873
recall : 0.33512568337728954
f1 : 0.31829304816162324
roc_auc : 0.6202925413504751
